# Gene annotation and gene profiling



In [14]:
from google.colab import drive
drive.mount('/content/drive')
py_env='/content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/python_env'
root_dir = "/content/drive/MyDrive/workshop_June2023"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## The workflow:

*   Bacterial gene prediction: prodigal
*   Generating the gene catalog: cd-hit
*   Building a bowtie2 database from the gene catalog: bowtie2
*   Mapping short reads against the gene catalog: bowtie2
*   Calculating the coverage for each sample

## Setup of the environment



In [ ]:
# conda environment
import os,sys
sys.path.append(py_env)
import condacolab
condacolab.install()

# for gene call
! conda install -c bioconda prodigal
# for cd-hit
! conda install -c bioconda cd-hit # 41s
# for gene profiling
! sudo apt install bowtie2 # setup bowtie2
!conda install -c bioconda samtools>1.10 # ~ 5 minutes


## Check the installation was successful

In [12]:
! which prodigal
! which samtools
! which bowtie2
! which cd-hit

/usr/local/bin/prodigal
/usr/local/bin/samtools
/usr/bin/bowtie2
/usr/local/bin/cd-hit


## Predict bacterial genes on contigs

In [ ]:
import os
gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call_demo")
contig_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","contigs")
sample_id = "PSMB4MBK"

! mkdir -p {gene_call_dir}
output_gff = os.path.join(gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(gene_call_dir,sample_id+".prodigal.fna")
! ls -lh {contig_dir}/{sample_id}_contigs.fna

# run gene prediction
! prodigal -p meta -i {contig_dir}/{sample_id}_contigs.fna -f gff -o {output_gff} -a {output_faa} -d {output_fna} # take 7 min for run

## Overview of the output


In [16]:
demo_gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call_demo")
# pre-calculated files
output_gff = os.path.join(demo_gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(demo_gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(demo_gene_call_dir,sample_id+".prodigal.fna")
! head {output_fna}

>k105_1_1 # 3 # 107 # -1 # ID=1_1;partial=10;start_type=ATG;rbs_motif=None;rbs_spacer=None;gc_cont=0.429
ATGGATGAACTGACCGACATTTACAAAAGAATAGAATACCTGCGCAACAATGGCGTAAAGATGAAAGAAA
TTGCCGACCGTGTAGATATGGCACCAAGCGTATTG
>k105_2_1 # 29 # 2416 # -1 # ID=2_1;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.371
TCCATTACAGTGCGTAAATTGGGGCGCATGCCTAAAATCGTAAAAGATCCTTATATACAGGCTAGCTATA
AAAAAATTATGGGTAAGCCTTGGTACGATCTTTATTCAGACGAGGATTTAAAATACATTGAAAAGATGAA
AGAAGACCCAACCTTACCTAATGTGATTCCAAGTTTCAGTAATCCAGAATATTATACCTATTTAGGGAAT
ACAGACTGGTTTTCGGAAATATATGATAACACAGGTATAACTCACTCTCATAATTTAAGTTTGTCAGGCG
CTTCCGAAAAGGCTTCTTATTATATAGGTATGGAATATATGCAGGAAAGAGGGCTCTTAAAAATTAATAA
GGACATTATGGATCGTTATAATTTCCGCTCAAAAGTAGATTTCAAAGTAGCCGATTGGTTGACTTTTGGC


### Task: count all complete genes

Incomplete genes are marked by the "partial" field in the header:

*   A complete gene: **partial=00**
*   A gene that is incomplete on the left side: **partial=10**
*   A gene that is incomplete on the right side: **partial=01**
*   A gene that is incomplete on both sides: **partial=11**

Calculate the number of complete genes and the total number of genes.

Hint:


*   **grep** the headers with **partial=00**
*   count the results using the **wc** commend




In [ ]:
! grep "partial=00" {output_fna} | wc -l
! grep ">" {output_fna} | wc -l

26360
96758


### Put your solution here [2 min]


In [ ]:
# your solution here:


## Generating the gene catalog

A gene catalog consist of all possible, **non-redundent** protein-coding genes

Assuming that we have assembled genes from multiple samples, we are going to generate a gene catalog from these files.

We will start with a merged file with all the **complete** genes from multiple samples

In [17]:
demo_gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
merged_fa = os.path.join(demo_gene_call_dir,"demo_merged_genes.fna")
! head {merged_fa}

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [ ]:
coverage=90
identity=95

! cd-hit-est -i {merged_fa} -aS 0.{coverage} -aL 0.{coverage} -c 0.{identity} -M 0 -r 0 -B 0 -d 0 -o {demo_gene_call_dir}/nr.fa -sc 1;

### Overview of the result

There are two major output files:


1.   The representative sequence of each cluster [fasta format]
2.   A cluster file recording the cluster each gene originates from [.clsr]



In [19]:
! head {demo_gene_call_dir}/nr.fa

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [20]:
! head {demo_gene_call_dir}/nr.fa.clstr

>Cluster 0
0	117nt, >CSM5MCXT_k105_7934::2::331::447::-... *
1	117nt, >CSM5MCXT_k105_39322::2::174::290::+... at +/97.44%
2	117nt, >CSM5MCXT_k105_41278::23::23895::24011::-... at +/95.73%
3	117nt, >CSM5MCW6_k105_30839::49::69293::69409::-... at +/95.73%
>Cluster 1
0	315nt, >CSM5MCXT_k105_1156::2::349::663::+... *
1	315nt, >CSM5MCXT_k105_14525::2::349::663::+... at +/96.83%
2	315nt, >CSM5MCW6_k105_8476::1::62::376::-... at +/98.10%
>Cluster 2


### Task: count the number of resulting clusters

In [ ]:
# your solution here:
! tail {demo_gene_call_dir}/nr.fa.clstr

>Cluster 18987
0	858nt, >CSM5MCXT_k105_6584::2::296::1153::+... *
>Cluster 18988
0	858nt, >CSM5MCXT_k105_16943::6::6402::7259::-... *
>Cluster 18989
0	858nt, >CSM5MCXT_k105_17213::1::53::910::+... *
>Cluster 18990
0	858nt, >CSM5MCXT_k105_14769::5::3299::4156::+... *
>Cluster 18991
0	858nt, >CSM5MCXT_k105_16395::3::2850::3707::-... *


In [ ]:
! grep ">Cluster" {demo_gene_call_dir}/nr.fa.clstr | wc -l

18992


## Create a bowtie2 database from the gene catalog



In [ ]:
# build bowtie2 database. 1 min for demo
! bowtie2-build {demo_gene_call_dir}/nr.fa {demo_gene_call_dir}/nr.fa_bowtie2DB

In [22]:
! ls -lh {demo_gene_call_dir}/nr.fa_bowtie2DB*

-rw------- 1 root root  11M Jun 16 09:42 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.1.bt2
-rw------- 1 root root 3.8M Jun 16 09:42 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.2.bt2
-rw------- 1 root root 167K Jun 16 09:41 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.3.bt2
-rw------- 1 root root 3.8M Jun 16 09:41 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.4.bt2
-rw------- 1 root root  11M Jun 16 09:42 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.rev.1.bt2
-rw------- 1 root root 3.8M Jun 16 09:42 /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.rev.2.bt2


# Profile the abundance of each gene in each sample

**Workflow**


1.   Map the reads against the bowtie2 database
2.   Calculate the coverage of each gene





In [23]:
import os
mgx_reads_dir = os.path.join(root_dir,"mgx_reads")
output_sam_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_gf_mapping","raw")
! mkdir -p {output_sam_dir}
sample_id="PSMB4MBK"
# p1 = os.path.join(mgx_reads_dir,sample_id+"_R1.fastq.gz")
# p2 = os.path.join(mgx_reads_dir,sample_id+"_R2.fastq.gz")
! ls -lh /content/drive/MyDrive/workshop_June2023/mgx_reads

# here we do subset:
# ! zcat /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R1.fastq.gz | head -n 40000 > /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R1.fastq
# ! gzip /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R1.fastq
# ! zcat /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R2.fastq.gz | head -n 40000 > /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R2.fastq
# ! gzip /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R2.fastq

p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")

! bowtie2 -x {demo_gene_call_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset

total 780M
-rw------- 1 root root  15K May  3 04:55 PSMB4MBK.log
-rw------- 1 root root 389M May  3 04:55 PSMB4MBK_R1.fastq.gz
-rw------- 1 root root 391M May  3 04:55 PSMB4MBK_R2.fastq.gz
-rw------- 1 root root 482K Jun 12 14:25 PSMB4MBK_subset_R1.fastq.gz
-rw------- 1 root root 478K Jun 12 14:27 PSMB4MBK_subset_R2.fastq.gz
10000 reads; of these:
  10000 (100.00%) were paired; of these:
    9365 (93.65%) aligned concordantly 0 times
    569 (5.69%) aligned concordantly exactly 1 time
    66 (0.66%) aligned concordantly >1 times
    ----
    9365 pairs aligned concordantly 0 times; of these:
      37 (0.40%) aligned discordantly 1 time
    ----
    9328 pairs aligned 0 times concordantly or discordantly; of these:
      18656 mates make up the pairs; of these:
        18567 (99.52%) aligned 0 times
        74 (0.40%) aligned exactly 1 time
        15 (0.08%) aligned >1 times
7.17% overall alignment rate


### Profiling the coverage from the sam file

In [24]:
! which samtools
! samtools --version
! samtools coverage --help

/usr/local/bin/samtools
samtools 1.17
Using htslib 1.17
Copyright (C) 2023 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             /opt/conda/conda-bld/samtools_1684314010921/_build_env/bin/x86_64-conda-linux-gnu-cc
    CPPFLAGS:       -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /usr/local/include
    CFLAGS:         -Wall -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /usr/local/include -fdebug-prefix-map=/opt/conda/conda-bld/samtools_1684314010921/work=/usr/local/src/conda/samtools-1.17 -fdebug-prefix-map=/usr/local=/usr/local/src/conda-prefix
    LDFLAGS:        -Wl,-O2 -Wl,--sort-common -Wl,--as-needed -Wl,-z,relro -Wl,-z,now -Wl,--disable-new-dtags -Wl,--gc-sections -Wl,--allow-shlib-undefined -Wl,-rpath,/usr/local/lib -Wl,-rpath-link,/usr/local/lib -L/usr/local/lib
    HTSDIR:         
    LIBS:           
    CURSES_LIB:     -ltinfow -lncurs

In [25]:
# sort the sam file
!samtools sort {output_sam_dir}/{sample_id}.sam -o {output_sam_dir}/{sample_id}.sorted_bam # 3 min
# generate the coverage file
!samtools coverage {output_sam_dir}/{sample_id}.sorted_bam > {output_sam_dir}/{sample_id}.coverage.txt

In [ ]:
# check the output:
! ls -lh /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw
! head {output_sam_dir}/{sample_id}.coverage.txt

total 8.2M
-rw------- 1 root root 1.1M Jun 12 14:34 PSMB4MBK.coverage.txt
-rw------- 1 root root 5.8M Jun 12 14:30 PSMB4MBK.sam
-rw------- 1 root root 1.4M Jun 12 14:34 PSMB4MBK.sorted_bam
#rname	startpos	endpos	numreads	covbases	coverage	meandepth	meanbaseq	meanmapq
CSM5MCXT_k105_124::1::29::946::+	1	918	2	52	5.66449	0.11329	36	42
CSM5MCXT_k105_164::3::1144::2457::-	1	1314	2	79	6.01218	0.120244	36.4	42
CSM5MCXT_k105_255::2::238::2178::-	1	1941	2	196	10.0979	0.10407	36.6	42
CSM5MCXT_k105_255::9::9113::10072::-	1	960	2	63	6.5625	0.13125	36.7	42
CSM5MCXT_k105_255::14::11836::15096::+	1	3261	2	141	4.32383	0.0619442	36.2	42
CSM5MCXT_k105_255::46::51602::53176::+	1	1575	1	101	6.4127	0.064127	36.6	42
CSM5MCXT_k105_255::63::74480::75271::+	1	792	2	123	15.5303	0.255051	36.7	42
CSM5MCXT_k105_255::68::80391::83516::+	1	3126	2	85	2.71913	0.0543826	34.8	42
CSM5MCXT_k105_296::2::181::1527::-	1	1347	20	828	61.4699	1.39569	36.2	38.4
